In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
data_path = "/kaggle/input/datasets/thienmaivan/india-pines/Indian_pines_corrected.mat"
gt_path   = "/kaggle/input/datasets/thienmaivan/india-pines/Indian_pines_gt.mat"

In [ ]:
from scipy.io import loadmat
import numpy as np

data_mat = loadmat(data_path)
gt_mat = loadmat(gt_path)

print("Data keys:", data_mat.keys())
print("GT keys:", gt_mat.keys())

In [ ]:
X = data_mat["indian_pines_corrected"]
y = gt_mat["indian_pines_gt"]

print("HSI shape:", X.shape)
print("GT shape :", y.shape)

print("HSI dtype:", X.dtype)
print("GT dtype :", y.dtype)

print("Labels:", np.unique(y))

In [ ]:
class_names = [
    "Alfalfa",
    "Corn-notill",
    "Corn-mintill",
    "Corn",
    "Grass-pasture",
    "Grass-trees",
    "Grass-pasture-mowed",
    "Hay-windrowed",
    "Oats",
    "Soybean-notill",
    "Soybean-mintill",
    "Soybean-clean",
    "Wheat",
    "Woods",
    "Buildings-Grass-Trees-Drives",
    "Stone-Steel-Towers"
]

total_labeled = np.sum(y > 0)

print("Tổng pixel có nhãn:", total_labeled)
print()

for cls in range(1, 17):
    count = np.sum(y == cls)
    percent = count / total_labeled * 100

    print(
        f"Class {cls:2d} | "
        f"{class_names[cls-1]:30s} | "
        f"{count:4d} samples | "
        f"{percent:6.2f}%"
    )

In [ ]:
import matplotlib.pyplot as plt

counts = [
    np.sum(y == cls)
    for cls in range(1, 17)
]

plt.figure(figsize=(12, 5))

plt.bar(
    range(1, 17),
    counts
)

plt.xticks(
    range(1, 17)
)

plt.xlabel("Class")
plt.ylabel("Number of labeled pixels")
plt.title("Indian Pines - Class Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

plt.imshow(y, cmap="nipy_spectral")

plt.title("Indian Pines - Ground Truth Map")
plt.colorbar(label="Class label")

plt.axis("off")
plt.show()

In [ ]:
bands = [10, 50, 100, 150]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, band in zip(axes, bands):
    ax.imshow(X[:, :, band], cmap="gray")
    ax.set_title(f"Band {band}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
coords = np.argwhere(y > 0)

row, col = coords[1000]

spectrum = X[row, col, :]

plt.figure(figsize=(10, 4))
plt.plot(spectrum)

plt.xlabel("Band")
plt.ylabel("Reflectance value")
plt.title(
    f"Spectral signature at ({row}, {col}) - Class {y[row, col]}"
)

plt.show()

In [ ]:
# Chọn một pixel có nhãn
coords = np.argwhere(y > 0)

row, col = coords[1000]

spectrum = X[row, col, :]

print("Pixel:", (row, col))
print("Class:", y[row, col])
print("Spectrum shape:", spectrum.shape)
print("First 10 band values:", spectrum[:10])

plt.figure(figsize=(10, 4))
plt.plot(spectrum)

plt.xlabel("Band index")
plt.ylabel("Reflectance value")
plt.title(
    f"Spectral signature - Pixel ({row}, {col}) - Class {y[row, col]}"
)

plt.show()

Từng vùng phổ tại mỗi pixel có sự phản ứng khác nhau

In [ ]:
# Chọn class cần quan sát
class_a = 3
class_b = 11

# Lấy tọa độ các pixel thuộc từng class
coords_a = np.argwhere(y == class_a)
coords_b = np.argwhere(y == class_b)

# Chọn 2 pixel thuộc class A
row_a1, col_a1 = coords_a[0]
row_a2, col_a2 = coords_a[len(coords_a) // 2]

# Chọn 1 pixel thuộc class B
row_b1, col_b1 = coords_b[0]

# Lấy spectral signature của từng pixel
spectrum_a1 = X[row_a1, col_a1, :]
spectrum_a2 = X[row_a2, col_a2, :]
spectrum_b1 = X[row_b1, col_b1, :]

# In thông tin pixel
print("Pixel A1:", (row_a1, col_a1), "Class:", y[row_a1, col_a1])
print("Pixel A2:", (row_a2, col_a2), "Class:", y[row_a2, col_a2])
print("Pixel B1:", (row_b1, col_b1), "Class:", y[row_b1, col_b1])

# Vẽ 3 spectral signature trên cùng một biểu đồ
plt.figure(figsize=(12, 5))

plt.plot(
    spectrum_a1,
    label=f"Class {class_a} - Pixel ({row_a1}, {col_a1})"
)

plt.plot(
    spectrum_a2,
    label=f"Class {class_a} - Pixel ({row_a2}, {col_a2})"
)

plt.plot(
    spectrum_b1,
    label=f"Class {class_b} - Pixel ({row_b1}, {col_b1})"
)

plt.xlabel("Band index")
plt.ylabel("Pixel value")
plt.title("Comparison of spectral signatures")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# Hai class cần so sánh
class_a = 3
class_b = 11

# Lấy toàn bộ spectral vectors của mỗi class
# Kết quả có dạng: [số_pixel, 200_band]
spectra_a = X[y == class_a]
spectra_b = X[y == class_b]

# Tính phổ trung bình của mỗi class
mean_a = spectra_a.mean(axis=0)
mean_b = spectra_b.mean(axis=0)

# Tính độ lệch chuẩn để xem mức dao động trong cùng class
std_a = spectra_a.std(axis=0)
std_b = spectra_b.std(axis=0)

print("Class 3 samples :", len(spectra_a))
print("Class 11 samples:", len(spectra_b))

# Vẽ phổ trung bình
plt.figure(figsize=(12, 5))

plt.plot(mean_a, label="Class 3 - Mean spectrum")
plt.plot(mean_b, label="Class 11 - Mean spectrum")

# Vùng dao động ±1 độ lệch chuẩn
plt.fill_between(
    np.arange(200),
    mean_a - std_a,
    mean_a + std_a,
    alpha=0.15
)

plt.fill_between(
    np.arange(200),
    mean_b - std_b,
    mean_b + std_b,
    alpha=0.15
)

plt.xlabel("Band index")
plt.ylabel("Pixel value")
plt.title("Mean spectral signatures of Class 3 and Class 11")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:


# Kích thước patch phải là số lẻ để có đúng 1 pixel trung tâm
patch_size = 5
half = patch_size // 2   # 5 // 2 = 2

# Lấy tất cả tọa độ pixel có nhãn
coords = np.argwhere(y > 0)

# Chọn thử một pixel có nhãn
row, col = coords[1000]

# Nhãn của pixel trung tâm
label = y[row, col]

# Padding ảnh HSI để những pixel nằm gần biên vẫn lấy được patch đầy đủ
# Chỉ padding theo 2 chiều không gian, không padding chiều band
X_padded = np.pad(
    X,
    ((half, half), (half, half), (0, 0)),
    mode="reflect"
)

# Vì đã padding nên tọa độ phải dịch thêm 'half'
row_p = row + half
col_p = col + half

# Lấy vùng 5x5 xung quanh pixel trung tâm
# Dấu ":" cuối cùng nghĩa là lấy toàn bộ 200 band
patch = X_padded[
    row_p - half : row_p + half + 1,
    col_p - half : col_p + half + 1,
    :
]

print("Pixel trung tâm:", (row, col))
print("Class:", label)
print("Patch shape:", patch.shape)

# Kiểm tra spectral vector của pixel trung tâm trong patch
center_spectrum = patch[half, half, :]

print("Center spectrum shape:", center_spectrum.shape)
print("First 10 band values:", center_spectrum[:10])

In [ ]:
band = 50

plt.figure(figsize=(4, 4))

plt.imshow(
    patch[:, :, band],
    cmap="gray"
)

plt.title(
    f"5x5 patch - Band {band}\n"
    f"Center pixel: ({row}, {col}) - Class {label}"
)

plt.colorbar(label="Pixel value")
plt.show()

In [ ]:


# Chọn các band muốn quan sát
bands = [10, 50, 100, 150]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, band in zip(axes, bands):
    # Lấy lát cắt 5x5 của patch tại band đang xét
    patch_band = patch[:, :, band]

    ax.imshow(patch_band, cmap="gray")
    ax.set_title(f"Band {band}")
    ax.axis("off")

plt.suptitle(
    f"Same 5x5 patch at different bands\n"
    f"Center pixel: ({row}, {col}) - Class {label}"
)

plt.tight_layout()
plt.show()
